In [1]:
import os, math, random
import numpy as np, pandas as pd, cv2, matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED=42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
PROJECT_ROOT=os.path.abspath("..")  # notebook is in notebooks/
DATA_PROCESSED=os.path.join(PROJECT_ROOT,"data","processed")
MODEL_DIR=os.path.join(PROJECT_ROOT,"models")
EXPERIMENT_DIR=os.path.join(PROJECT_ROOT,"experiments","training")
os.makedirs(MODEL_DIR,exist_ok=True); os.makedirs(EXPERIMENT_DIR,exist_ok=True)
IMG_HEIGHT,IMG_WIDTH=66,200
BATCH_SIZE=64
EPOCHS=40
CAMERA_STEERING_CORRECTION=0.20
print("TensorFlow:",tf.__version__,"GPUs:",tf.config.list_physical_devices("GPU"))
print("Project root:",PROJECT_ROOT)

TensorFlow: 2.21.0 GPUs: []
Project root: C:\Users\user\Neuro Drive


In [4]:
train_df=pd.read_csv(os.path.join(DATA_PROCESSED,"train_manifest.csv"))
val_data=np.load(os.path.join(DATA_PROCESSED,"validation_data.npz"))
X_val=val_data["X_val"]
y_val=val_data["y_val"].astype(np.float32).reshape(-1)
print("Train rows:",len(train_df),"Validation:",X_val.shape,y_val.shape)
display(train_df.head())
assert len(train_df)>0 and len(X_val)>0
assert X_val.shape[1:]==(IMG_HEIGHT,IMG_WIDTH,3)

Train rows: 2037 Validation: (437, 66, 200, 3) (437,)


,center,left,right,steering,sampling_weight
0,C:\Users\user\Neuro Drive\data\raw\IMG\center_...,C:\Users\user\Neuro Drive\data\raw\IMG\left_20...,C:\Users\user\Neuro Drive\data\raw\IMG\right_2...,0.0,0.079184
1,C:\Users\user\Neuro Drive\data\raw\IMG\center_...,C:\Users\user\Neuro Drive\data\raw\IMG\left_20...,C:\Users\user\Neuro Drive\data\raw\IMG\right_2...,1.0,0.462955
2,C:\Users\user\Neuro Drive\data\raw\IMG\center_...,C:\Users\user\Neuro Drive\data\raw\IMG\left_20...,C:\Users\user\Neuro Drive\data\raw\IMG\right_2...,-0.2,2.037000
3,C:\Users\user\Neuro Drive\data\raw\IMG\center_...,C:\Users\user\Neuro Drive\data\raw\IMG\left_20...,C:\Users\user\Neuro Drive\data\raw\IMG\right_2...,-1.0,0.389856
4,C:\Users\user\Neuro Drive\data\raw\IMG\center_...,C:\Users\user\Neuro Drive\data\raw\IMG\left_20...,C:\Users\user\Neuro Drive\data\raw\IMG\right_2...,0.0,0.079184


In [10]:
def preprocess_image(image):
    h,w=image.shape[:2]
    image=image[int(h*.35):h-int(h*.10),:]
    image=cv2.resize(image,(IMG_WIDTH,IMG_HEIGHT),interpolation=cv2.INTER_AREA)
    return cv2.cvtColor(image,cv2.COLOR_BGR2RGB)

def augment_brightness(image):
    factor=np.random.uniform(.65,1.35)
    return np.clip(image.astype(np.float32)*factor,0,255).astype(np.uint8)

def augment_flip(image,steering):
    if np.random.rand()<.5:
        image=cv2.flip(image,1); steering=-steering
    return image,steering

def load_training_sample(row):
    camera=np.random.choice(["center","left","right"])
    path=str(row[f"{camera}"])
    steering=float(row["steering"])
    if camera=="left": steering+=CAMERA_STEERING_CORRECTION
    elif camera=="right": steering-=CAMERA_STEERING_CORRECTION
    image=cv2.imread(path)
    if image is None: raise FileNotFoundError(path)
    image=preprocess_image(image)
    image=augment_brightness(image)
    image,steering=augment_flip(image,steering)
    return image,np.float32(steering)

In [11]:
class DrivingSequence(keras.utils.Sequence):
    def __init__(self,dataframe,batch_size=64,shuffle=True,**kwargs):
        super().__init__(**kwargs)
        self.df=dataframe.reset_index(drop=True).copy()
        self.batch_size=batch_size; self.shuffle=shuffle
        self.indices=np.arange(len(self.df)); self.on_epoch_end()
    def __len__(self):
        return math.ceil(len(self.indices)/self.batch_size)
    def __getitem__(self,batch_index):
        ids=self.indices[batch_index*self.batch_size:(batch_index+1)*self.batch_size]
        samples=[load_training_sample(self.df.iloc[i]) for i in ids]
        images,angles=zip(*samples)
        return np.asarray(images,dtype=np.uint8),np.asarray(angles,dtype=np.float32)
    def on_epoch_end(self):
        if self.shuffle: np.random.shuffle(self.indices)

train_sequence=DrivingSequence(train_df,BATCH_SIZE,True)
print("Batches per epoch:",len(train_sequence))

Batches per epoch: 32


In [12]:
def build_neurodrive_model():
    inputs=keras.Input((IMG_HEIGHT,IMG_WIDTH,3),name="camera_image")
    x=layers.Rescaling(1/255.0,offset=-.5,name="normalize")(inputs)
    x=layers.Conv2D(24,5,strides=2,activation="relu",name="conv1")(x)
    x=layers.Conv2D(36,5,strides=2,activation="relu",name="conv2")(x)
    x=layers.Conv2D(48,5,strides=2,activation="relu",name="conv3")(x)
    x=layers.Conv2D(64,3,activation="relu",name="conv4")(x)
    x=layers.Conv2D(64,3,activation="relu",name="conv5")(x)
    x=layers.Flatten()(x)
    x=layers.Dense(100,activation="relu")(x)
    x=layers.Dense(50,activation="relu")(x)
    x=layers.Dense(10,activation="relu")(x)
    outputs=layers.Dense(1,activation="linear",name="steering")(x)
    return keras.Model(inputs,outputs,name="NeuroDriveCNN")

model=build_neurodrive_model()
model.summary()

Model: "NeuroDriveCNN"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ camera_image (InputLayer)            │ (None, 66, 200, 3)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ normalize (Rescaling)                │ (None, 66, 200, 3)          │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv1 (Conv2D)                       │ (None, 31, 98, 24)          │           1,824 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv2 (Conv2D)                       │ (None, 14, 47, 36)          │          21,636 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv3 (Conv2D)                       │ (None, 5, 22, 48)           │          43,248 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv4 (Conv2D)                       │ (None, 3, 20, 64)           │          27,712 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ conv5 (Conv2D)                       │ (None, 1, 18, 64)           │          36,928 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ flatten_1 (Flatten)                  │ (None, 1152)                │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_3 (Dense)                      │ (None, 100)                 │         115,300 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_4 (Dense)                      │ (None, 50)                  │           5,050 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 10)                  │             510 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ steering (Dense)                     │ (None, 1)                   │              11 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 252,219 (985.23 KB)

 Trainable params: 252,219 (985.23 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-4),
              loss="mse",metrics=[keras.metrics.MeanAbsoluteError(name="mae")])
callbacks=[
 keras.callbacks.ModelCheckpoint(os.path.join(MODEL_DIR,"neurodrive_best.keras"),
                                monitor="val_loss",save_best_only=True,verbose=1),
 keras.callbacks.EarlyStopping(monitor="val_loss",patience=8,restore_best_weights=True,verbose=1),
 keras.callbacks.ReduceLROnPlateau(monitor="val_loss",factor=.5,patience=3,min_lr=1e-7,verbose=1),
 keras.callbacks.CSVLogger(os.path.join(EXPERIMENT_DIR,"training_log.csv"))
]

In [ ]:
history=model.fit(train_sequence,validation_data=(X_val,y_val),
                  epochs=EPOCHS,callbacks=callbacks,verbose=1)

Epoch 1/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 439ms/step - loss: 0.2791 - mae: 0.3823
Epoch 1: val_loss improved from None to 0.22396, saving model to C:\Users\user\Neuro Drive\models\neurodrive_best.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 16s 454ms/step - loss: 0.2795 - mae: 0.3878 - val_loss: 0.2240 - val_mae: 0.3409 - learning_rate: 1.0000e-04
Epoch 2/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 304ms/step - loss: 0.2476 - mae: 0.3806
Epoch 2: val_loss improved from 0.22396 to 0.20540, saving model to C:\Users\user\Neuro Drive\models\neurodrive_best.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 10s 310ms/step - loss: 0.2491 - mae: 0.3802 - val_loss: 0.2054 - val_mae: 0.3443 - learning_rate: 1.0000e-04
Epoch 3/40
32/32 ━━━━━━━━━━━━━━━━━━━━ 0s 253ms/step - loss: 0.2349 - mae: 0.3744
Epoch 3: val_loss improved from 0.20540 to 0.19699, saving model to C:\Users\user\Neuro Drive\models\neurodrive_best.keras
32/32 ━━━━━━━━━━━━━━━━━━━━ 8s 261ms/step - loss: 0.2362 - mae: 0.3748 - val_loss: 0.1970 - val_mae: 0.3368 - learning_r